# Testing the `BranchedCoverHomology` class

## Setup

In [1]:
import path
from gaknot_lib import notebook_setup
import ipytest
import pytest
import warnings

# Suppress DeprecationWarnings from SageMath and other libraries
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message=".*superseded by LazyCombinatorialSpecies.*")
warnings.filterwarnings('ignore', message=".*Importing .* from here is deprecated.*")

ipytest.autoconfig(raise_on_error=True)

The ipytest module is not an IPython extension.
INFO: 

import_sage called with arguments:
	module_name: signature
	package: gaknot_lib
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
INFO: 

import_sage called with arguments:
	module_name: LT_signature
	package: gaknot_lib
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
INFO: 

import_sage called with arguments:
	module_name: signature
	package: gaknot_lib
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
INFO: 

import_sage called with arguments:
	module_name: gaknot
	package: gaknot_lib
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
INFO: 

import_sage called with arguments:
	module_name: H1_branched_cover
	package: gaknot_lib
	path: /User

In [2]:
%preparse gaknot_lib
%preparse H1_branched_cover
from gaknot_lib.gaknot import GeneralizedAlgebraicKnot
from gaknot_lib.H1_branched_cover import BranchedCoverHomology
from sage.all import ZZ, gcd

INFO: 

import_sage called with arguments:
	module_name: gaknot
	package: gaknot_lib
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
Successfully preparsed and reloaded: gaknot
INFO: 

import_sage called with arguments:
	module_name: H1_branched_cover
	package: gaknot_lib
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
Successfully preparsed and reloaded: H1_branched_cover


## Tests

### Basic Torus Knots
Verifies the homology computation for $N$-fold branched covers of simple torus knots $T(p,q)$.

This test verifies that the `BranchedCoverHomology` class correctly computes the homology of $N$-fold branched covers for various simple torus knots $T(p,q)$. It checks both the invariant factors and the human-readable string representation of the resulting homology group.

In [3]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("p, q, n, expected_factors, expected_str", [
    (2, 3, 2, [3], "(Z/3Z)[T(2,3)]"),
    (2, 3, 3, [2, 2], "(Z/2Z ⊕ Z/2Z)[T(2,3)]"),
    (2, 5, 2, [5], "(Z/5Z)[T(2,5)]"),
    (2, 5, 4, [5], "(Z/5Z)[T(2,5)]"),
    (3, 4, 2, [3], "(Z/3Z)[T(3,4)]"),
    (3, 4, 3, [4, 4], "(Z/4Z ⊕ Z/4Z)[T(3,4)]"),
    (2, 7, 2, [7], "(Z/7Z)[T(2,7)]"),
    (3, 5, 2, [], "0"),
    (2, 3, 6, [0, 0], "(Z ⊕ Z)[T(2,3)]"),
    (5, 2, 2, [5], "(Z/5Z)[T(5,2)]")
])
def test_h1_torus_knot_parametric(p, q, n, expected_factors, expected_str):
    knot = GeneralizedAlgebraicKnot([(1, [(p, q)])])
    h1 = BranchedCoverHomology(knot, n)
    assert len(h1) == 1
    comp = h1[0]
    factors = [f for f in comp['layers'][0]['base_factors'] if f != 1]
    expected_non_trivial = [f for f in expected_factors if f != 1]
    assert factors == expected_non_trivial
    assert str(h1) == expected_str

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_6225ab1457184525af6a1c8924ca5722.py::test_h1_torus_knot_parametric[p0-q0-n0-expected_factors0-(Z/3Z)[T(2,3)]] PASSED [ 10%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_torus_knot_parametric[p1-q1-n1-expected_factors1-(Z/2Z \u2295 Z/2Z)[T(2,3)]] PASSED [ 20%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_torus_knot_parametric[p2-q2-n2-expected_factors2-(Z/5Z)[T(2,5)]] PASSED [ 30%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_torus_knot_parametric[p3-q3-n3-expected_factors3-(Z/5Z)[T(2,5)]] PASSED [ 40%]
t_6225ab1457184525af6

### Iterated Torus Knots
Tests the recursive homology decomposition for cabling structures.

This test focuses on iterated torus knots (satellites). It verifies the recursive application of Litherland's formula by checking the parameters, base factors, and multiplicities for each layer in the decomposition. It also ensures that the total invariant factors of the cover are correctly computed.

In [4]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("desc, n, expected_layers, expected_inv_factors", [
    ([(2, 3), (2, 5)], 2, 
     [{'params': (2, 5), 'base': [5], 'mult': 1}, {'params': (2, 3), 'base': [], 'mult': 2}], [5]),
    ([(2, 3), (6, 5)], 2,
     [{'params': (6, 5), 'base': [5], 'mult': 1}, {'params': (2, 3), 'base': [], 'mult': 2}], [5]),
    ([(2, 3), (2, 5)], 3,
     [{'params': (2, 5), 'base': [], 'mult': 1}, {'params': (2, 3), 'base': [2, 2], 'mult': 1}], [2, 2]),
    ([(2, 5), (2, 3)], 2,
     [{'params': (2, 3), 'base': [3], 'mult': 1}, {'params': (2, 5), 'base': [], 'mult': 2}], [3]),
    ([(3, 4), (2, 3)], 2,
     [{'params': (2, 3), 'base': [3], 'mult': 1}, {'params': (3, 4), 'base': [], 'mult': 2}], [3]),
    ([(2, 3), (2, 7)], 2,
     [{'params': (2, 7), 'base': [7], 'mult': 1}, {'params': (2, 3), 'base': [], 'mult': 2}], [7]),
    ([(2, 3), (6, 7)], 2,
     [{'params': (6, 7), 'base': [7], 'mult': 1}, {'params': (2, 3), 'base': [], 'mult': 2}], [7]),
    ([(3, 2), (2, 3)], 2,
     [{'params': (2, 3), 'base': [3], 'mult': 1}, {'params': (3, 2), 'base': [], 'mult': 2}], [3]),
    ([(2, 3), (2, 5), (2, 7)], 2,
     [{'params': (2, 7), 'base': [7], 'mult': 1}, {'params': (2, 5), 'base': [], 'mult': 2}, {'params': (2, 3), 'base': [], 'mult': 2}], [7]),
    ([(2, 3), (2, 9)], 2,
     [{'params': (2, 9), 'base': [9], 'mult': 1}, {'params': (2, 3), 'base': [], 'mult': 2}], [9])
])
def test_h1_iterated_torus_knot_parametric(desc, n, expected_layers, expected_inv_factors):
    knot = GeneralizedAlgebraicKnot([(1, desc)])
    h1 = BranchedCoverHomology(knot, n)
    comp = h1[0]
    layers = comp['layers']
    
    for i, exp in enumerate(expected_layers):
        assert layers[i]['parameters'] == exp['params']
        assert [f for f in layers[i]['base_factors'] if f != 1] == [f for f in exp['base'] if f != 1]
        assert layers[i]['multiplicity'] == exp['mult']
    
    assert [f for f in h1.invariant_factors if f != 1] == sorted([f for f in expected_inv_factors if f != 1])

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_6225ab1457184525af6a1c8924ca5722.py::test_h1_iterated_torus_knot_parametric[desc0-n0-expected_layers0-expected_inv_factors0] PASSED [ 10%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_iterated_torus_knot_parametric[desc1-n1-expected_layers1-expected_inv_factors1] PASSED [ 20%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_iterated_torus_knot_parametric[desc2-n2-expected_layers2-expected_inv_factors2] PASSED [ 30%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_iterated_torus_knot_parametric[desc3-n3-expected_layers3-expected_i

### Connected Sums
Verifies that the homology of a connected sum is correctly calculated as the direct sum of its components' homologies.

This test confirms that the homology of a connected sum of knots is correctly calculated as the direct sum of the homologies of its individual components. It checks the number of components and the signs of the knot summands in the resulting homology decomposition.

In [5]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("sum_desc, n, expected_len, expected_signs", [
    ([(1, [(2, 3)]), (-1, [(2, 3), (2, 5)])], 2, 2, [1, -1]),
    ([(1, [(2, 3)]), (1, [(3, 4)])], 2, 2, [1, 1]),
    ([(-1, [(2, 5)]), (-1, [(2, 7)])], 2, 2, [-1, -1]),
    ([(1, [(2, 3)]), (1, [(2, 3)]), (1, [(2, 3)])], 2, 3, [1, 1, 1]),
    ([(1, [(2, 3), (6, 5)]), (-1, [(2, 3)])], 2, 2, [1, -1]),
    ([(1, [(2, 3)]), (-1, [(3, 4)]), (1, [(2, 5)])], 2, 3, [1, -1, 1]),
    ([(1, [(3, 2)]), (1, [(2, 3)])], 2, 2, [1, 1]),
    ([(1, [(2, 3)]), (-1, [(2, 3)])], 2, 2, [1, -1]),
    ([(1, [(2, 3), (2, 5), (2, 7)]), (1, [(3, 4)])], 2, 2, [1, 1]),
    ([(1, [(5, 7)]), (-1, [(5, 7)])], 2, 2, [1, -1])
])
def test_h1_connected_sum_parametric(sum_desc, n, expected_len, expected_signs):
    knot = GeneralizedAlgebraicKnot(sum_desc)
    h1 = BranchedCoverHomology(knot, n)
    assert len(h1) == expected_len
    for i, sign in enumerate(expected_signs):
        assert h1[i]['sign'] == sign

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_6225ab1457184525af6a1c8924ca5722.py::test_h1_connected_sum_parametric[sum_desc0-n0-expected_len0-expected_signs0] PASSED [ 10%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_connected_sum_parametric[sum_desc1-n1-expected_len1-expected_signs1] PASSED [ 20%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_connected_sum_parametric[sum_desc2-n2-expected_len2-expected_signs2] PASSED [ 30%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_connected_sum_parametric[sum_desc3-n3-expected_len3-expected_signs3] PASSED [ 40%]
t_6225ab145718452

### Homology Addition
Tests the overloaded `+` operator for `BranchedCoverHomology` objects.

This test verifies that the overloaded `+` operator for `BranchedCoverHomology` objects correctly combines the homology of two separate branched covers into a single group representing the connected sum of the underlying knots. It ensures the resulting decomposition is structurally sound.

In [6]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("desc1, desc2, n", [
    ([(1, [(2, 3)])], [(-1, [(2, 3), (2, 5)])], 2),
    ([(1, [(2, 5)])], [(1, [(2, 7)])], 2),
    ([(-1, [(3, 2)])], [(1, [(3, 4)])], 2),
    ([(1, [(2, 3), (6, 5)])], [(-1, [(2, 3)])], 2),
    ([(1, [(2, 3)])], [(1, [(2, 3)])], 3),
    ([(1, [(3, 4)])], [(1, [(4, 5)])], 2),
    ([(1, [(2, 3), (2, 5)])], [(1, [(2, 3), (2, 7)])], 2),
    ([(1, [(5, 2)])], [(-1, [(5, 3)])], 2),
    ([(1, [(2, 3)]), (1, [(3, 4)])], [(1, [(4, 5)])], 2),
    ([(1, [(2, 3)])], [(1, [(3, 4)]), (1, [(4, 5)])], 2)
])
def test_h1_addition_parametric(desc1, desc2, n):
    h1_1 = BranchedCoverHomology(GeneralizedAlgebraicKnot(desc1), n)
    h1_2 = BranchedCoverHomology(GeneralizedAlgebraicKnot(desc2), n)
    h1_sum = h1_1 + h1_2
    
    assert len(h1_sum) == len(h1_1) + len(h1_2)
    for i in range(len(h1_sum)):
        assert h1_sum[i]['index'] == i

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_6225ab1457184525af6a1c8924ca5722.py::test_h1_addition_parametric[desc10-desc20-n0] PASSED  [ 10%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_addition_parametric[desc11-desc21-n1] PASSED  [ 20%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_addition_parametric[desc12-desc22-n2] PASSED  [ 30%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_addition_parametric[desc13-desc23-n3] PASSED  [ 40%]
t_6225ab1457184525af6a1c8924ca5722.py::test_h1_addition_parametric[desc14-desc24-n4] PASSED  [ 50%]
t_6225ab1457184525af6a1c8924ca5722.py